<a href="https://colab.research.google.com/github/Ali-Hamza-developer/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check
**Lane: Refresh / Content Opportunity Scoring**

Run top to bottom (Runtime → Run all). Requires `HF_TOKEN` Colab Secret, same as w03.

In [1]:
!pip install -q duckdb

import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month — matches w03, never the sealed _sample
print('DuckDB ready, target month =', MONTH)

DuckDB ready, target month = 2026-03


## 1. Build the feature vector

Same 90-day feature window as w03's data contract (`month=2026-03`), rebuilt here as the canonical feature vector for this assignment: engineered features, categorical handling, and explicit fills for missing values.

In [2]:
feature_vector = con.sql(f"""
    WITH window_90d AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_90d,
            SUM(gsc_clicks) AS clicks_90d,
            AVG(gsc_avg_position) AS avg_position_90d,
            SUM(ga4_sessions) AS sessions_90d,
            SUM(ga4_engaged_sessions) AS engaged_sessions_90d
        FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        w.client_hash_id,
        w.content_hash_id,
        -- engineered numeric features
        LOG(1 + w.impressions_90d) AS log_impressions_90d,
        w.avg_position_90d,
        CASE WHEN w.impressions_90d > 0 THEN w.clicks_90d::DOUBLE / w.impressions_90d ELSE NULL END AS ctr_90d,
        CASE WHEN w.sessions_90d > 0 THEN w.engaged_sessions_90d::DOUBLE / w.sessions_90d ELSE NULL END AS engagement_rate_90d,
        (DATE '2026-03-31' - COALESCE(d.last_optimized_date, d.content_created_date)) AS days_since_last_update,
        d.word_count,
        -- categorical features, kept as strings for now — encode downstream
        d.content_type,
        d.main_intent,
        d.competition_level
    FROM window_90d w
    JOIN read_parquet('{REL}/dim_content.parquet') d
      ON w.content_hash_id = d.content_hash_id AND w.client_hash_id = d.client_hash_id
    WHERE w.impressions_90d > 0
      AND (DATE '2026-03-31' - COALESCE(d.last_optimized_date, d.content_created_date)) >= 0
""").df()

print('rows:', len(feature_vector))
print('missing values per column:')
print(feature_vector.isna().sum())
feature_vector.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 136974
missing values per column:
client_hash_id                0
content_hash_id               0
log_impressions_90d           0
avg_position_90d              0
ctr_90d                       0
engagement_rate_90d       96657
days_since_last_update        0
word_count                54711
content_type                  0
main_intent               15746
competition_level         16372
dtype: int64


,client_hash_id,content_hash_id,log_impressions_90d,avg_position_90d,ctr_90d,engagement_rate_90d,days_since_last_update,word_count,content_type,main_intent,competition_level
0,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2.657056,2.987198,0.000000,NaN,396,<NA>,keyword article,informational,MEDIUM
1,client_73cda7b4e4f265ea,content_1855a661b4d36130,2.633468,4.209227,0.002331,0.0,396,<NA>,keyword article,informational,HIGH
2,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,2.350248,9.445635,0.004484,0.0,396,<NA>,keyword article,informational,HIGH
3,client_73cda7b4e4f265ea,content_1f380a642aed423b,1.986772,6.014516,0.010417,0.0,396,<NA>,keyword article,commercial,HIGH
4,client_73cda7b4e4f265ea,content_22c063002b7c1caf,2.498311,9.155335,0.003185,NaN,396,<NA>,keyword article,informational,LOW


In [3]:
# Explicit fills — decide and document each one, don't silently dropna
feature_vector['ctr_90d'] = feature_vector['ctr_90d'].fillna(0.0)  # zero clicks despite impressions is a real, meaningful zero
feature_vector['engagement_rate_90d'] = feature_vector['engagement_rate_90d'].fillna(-1)  # -1 flags "no GA4 sessions this window" as distinct from "0% engagement"

# one-hot encode categoricals
feature_vector_encoded = pd.get_dummies(
    feature_vector,
    columns=['content_type', 'main_intent', 'competition_level'],
    dummy_na=True  # keep missing category as its own explicit column, don't silently drop
)

print('encoded shape:', feature_vector_encoded.shape)
feature_vector_encoded.head()

encoded shape: (136974, 21)


,client_hash_id,content_hash_id,log_impressions_90d,avg_position_90d,ctr_90d,engagement_rate_90d,days_since_last_update,word_count,content_type_comparison article,content_type_feedly article,...,content_type_nan,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_nan,competition_level_HIGH,competition_level_LOW,competition_level_MEDIUM,competition_level_nan
0,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2.657056,2.987198,0.000000,-1.0,396,<NA>,False,False,...,False,False,True,False,False,False,False,False,True,False
1,client_73cda7b4e4f265ea,content_1855a661b4d36130,2.633468,4.209227,0.002331,0.0,396,<NA>,False,False,...,False,False,True,False,False,False,True,False,False,False
2,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,2.350248,9.445635,0.004484,0.0,396,<NA>,False,False,...,False,False,True,False,False,False,True,False,False,False
3,client_73cda7b4e4f265ea,content_1f380a642aed423b,1.986772,6.014516,0.010417,0.0,396,<NA>,False,False,...,False,True,False,False,False,False,True,False,False,False
4,client_73cda7b4e4f265ea,content_22c063002b7c1caf,2.498311,9.155335,0.003185,-1.0,396,<NA>,False,False,...,False,False,True,False,False,False,False,True,False,False


## 2. Feature notes

| Feature | Meaning | Missing handling | Available before decision moment? |
|---|---|---|---|
| `log_impressions_90d` | log-scaled search impressions, prior 90 days | rows require impressions_90d > 0 by construction, so no NaN | Yes — strictly backward-looking window |
| `avg_position_90d` | average GSC search position, prior 90 days | NaN possible if no GSC rows in window; not yet filled — check before modeling | Yes |
| `ctr_90d` | clicks / impressions, prior 90 days | filled with 0.0 — impressions with zero clicks is a real signal, not missingness | Yes |
| `engagement_rate_90d` | engaged sessions / sessions, prior 90 days | filled with -1 sentinel — distinguishes "no GA4 tracking this window" from "0% engagement" | Yes, when GA4 tracking exists — check `client_has_ga4`/`ga4_data_available` before trusting this per-row |
| `days_since_last_update` | days between decision date (Mar 31) and last content optimization | rows with negative values (future update dates) were filtered out entirely in the query — see leakage hunt below | Yes, after the future-date filter |
| `word_count` | static content length | not expected to be missing; verify with `.isna().sum()` above | Yes — static metadata |
| `content_type`, `main_intent`, `competition_level` (one-hot) | categorical content/keyword attributes | `dummy_na=True` keeps missing as its own column instead of silently vanishing | Yes — static metadata, not tied to the outcome window |

## 3. The leakage hunt

Attack the feature vector deliberately: (a) a future-window column, (b) a label-derived column, (c) a rebuilt product flag. Show what each does to a quick score, then remove all three.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Build the label the same way as w03: 30-day forward window, month=2026-04
label_window = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_next30
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    WHERE report_date <= DATE '2026-04-30'
    GROUP BY 1, 2
""").df()

attack_df = feature_vector.merge(label_window, on=['client_hash_id', 'content_hash_id'], how='inner')
attack_df['future_decline'] = (
    attack_df['impressions_next30'] < 0.7 * (2**attack_df['log_impressions_90d'] - 1)
).astype(int)

# ATTACK 1 — future-window column, built directly from the label period
attack_df['next_30d_impressions_delta'] = (
    attack_df['impressions_next30'] - (2**attack_df['log_impressions_90d'] - 1)
)

# ATTACK 2 — label-derived column, near-copy of the target itself
attack_df['decline_flag_leak'] = attack_df['future_decline']  # obviously extreme, proves the point fast

# ATTACK 3 — a rebuilt product-style priority flag, built from CURRENT-window signals combined
# (not a future leak, but still excluded per lane guide — rebuilding a product decision flag
# and feeding it back in teaches the model to copy the old rule instead of finding real signal)
attack_df['rebuilt_priority_flag'] = (
    (attack_df['avg_position_90d'] > 10).astype(int) + (attack_df['ctr_90d'].fillna(0) < 0.02).astype(int)
)

features_honest = ['log_impressions_90d', 'avg_position_90d', 'word_count']
attack_configs = {
    'honest baseline': features_honest,
    '+ future-window leak': features_honest + ['next_30d_impressions_delta'],
    '+ label-derived leak': features_honest + ['decline_flag_leak'],
    '+ rebuilt product-flag leak': features_honest + ['rebuilt_priority_flag'],
}

results = {}
for name, feats in attack_configs.items():
    d = attack_df.dropna(subset=feats + ['future_decline'])
    X_train, X_test, y_train, y_test = train_test_split(d, d['future_decline'], test_size=0.3, random_state=42)
    clf = LogisticRegression(max_iter=1000).fit(X_train[feats], y_train)
    auc = roc_auc_score(y_test, clf.predict_proba(X_test[feats])[:, 1])
    results[name] = auc
    print(f"{name:32s} AUC = {auc:.3f}")

print("\nKeep only the honest baseline number for your model card — the rest exist to prove the leaks, not to report.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

honest baseline                  AUC = 0.881
+ future-window leak             AUC = 1.000
+ label-derived leak             AUC = 1.000
+ rebuilt product-flag leak      AUC = 0.881

Keep only the honest baseline number for your model card — the rest exist to prove the leaks, not to report.


**Fill in after running:** which attack inflated AUC the most, and why — in one sentence each. The label-derived leak should jump hardest (it's nearly a copy of the target); the future-window leak should also jump because it's built from the exact period being predicted; the rebuilt product-flag leak may or may not move AUC much — note whether it did, since a *small* jump there is still a reason to exclude it (circular reasoning, not necessarily magnitude).

## 4. What I excluded and why

| Excluded field | Why |
|---|---|
| `next_30d_impressions_delta` | built directly from the 30-day label window — demonstrated leak in Part 3 |
| `decline_flag_leak` / any near-copy of the target | trivially reconstructs the label — demonstrated leak in Part 3 |
| rebuilt product-style priority/health flags | not shipped in the release; rebuilding one and feeding it back causes a circular result (model learns to copy the old rule instead of finding independent signal) — demonstrated in Part 3 |
| `client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id` | join keys only, carry no signal of their own |
| raw query, URL, title, or client-identifying fields | not available in this release; would violate the public-safe output rules even if they were |
| any GA4 field on rows where `ga4_data_available IS FALSE` | not a real zero — absence of tracking, not absence of activity; using it as-is would misrepresent early-history rows as "no engagement" |

## 5. Self-check

- [ ] Every section filled — markdown thinking AND the code that backs it
- [ ] Notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to repo under `work/notebooks/w05_feature_leakage_check.ipynb` — then submit repo URL on the card